In [ ]:
# Colab setup: mount Google Drive and configure project paths.
# Run this cell first on Colab. If running locally, skip this cell.
from google.colab import drive
drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/Aini/ml-assignment/Team-Assignment-2/binus-ai-2026sem3-assignment2-group04"
MODEL_DIR   = f"{PROJECT_DIR}/models"
!mkdir -p "$MODEL_DIR"

import tensorflow as tf
print("GPU:", tf.config.list_physical_devices("GPU"))
print("Project dir:", PROJECT_DIR)
print("Model dir:  ", MODEL_DIR)

# Group Assignment 2 — Phase 1B: Transfer Learning with MobileNetV2

**Objective:**
Train a CIFAR-10 classifier using transfer learning from MobileNetV2 (ImageNet-pretrained), with the goal of substantially improving over the from-scratch VGG-style baseline (~87.4 % test accuracy from `01_baseline_EN.ipynb`).

**Strategy — two-stage training:**
1. **Stage 1 (frozen backbone, 10 epochs):** freeze all of MobileNetV2's pretrained weights, attach a fresh 10-class classifier head (`GlobalAveragePooling2D → Dropout(0.3) → Dense(10)`), train only the head with `lr=1e-3`. This adapts the head to CIFAR's class structure without disturbing the pretrained features.
2. **Stage 2 (fine-tune top 30 layers, 20 epochs):** unfreeze the deepest 30 layers of MobileNetV2 and continue training with a much smaller `lr=1e-5`. This nudges the high-level features toward CIFAR's specifics while preserving the early/mid-level representations.

**Why this is expected to outperform the baseline:**
MobileNetV2 was pretrained on ImageNet (14 M images at 224×224). Its low- and mid-level filters already encode generic visual concepts (edges, textures, simple shapes) that are far better than what 50 K CIFAR images can teach a from-scratch CNN. Transfer learning leverages this prior; we only have to learn the task-specific aspects.

**Required preprocessing departure from baseline:**
- CIFAR-10 (32×32) is **resized to 96×96** so MobileNetV2's filter scales activate properly. Below 96×96 the network refuses to build (Keras enforces a minimum input size).
- Pixel range is normalized via `tf.keras.applications.mobilenet_v2.preprocess_input` (maps [0, 255] → [-1, 1]), matching the distribution MobileNetV2 was pretrained on.

### Section 0: Setup and Reproducibility

We reuse `SEED = 42` from the baseline notebook so the train/validation split is identical between the two notebooks (same samples held out). Reproducibility caveats from the baseline still apply: weight initialization, dropout, and data augmentation are not seeded, so test accuracy will fluctuate by ±0.5 percentage points across runs.

In [ ]:
SEED = 42
IMG_SIZE = 96      # MobileNetV2 minimum supported spatial size
BATCH_SIZE = 64

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available:      {len(tf.config.list_physical_devices('GPU')) > 0}")
print(f"Random seed:        {SEED}")
print(f"Image size:         {IMG_SIZE}x{IMG_SIZE}")
print(f"Batch size:         {BATCH_SIZE}")

### Section 1: Load CIFAR-10 (Hugging Face)

We load CIFAR-10 via `datasets.load_dataset("cifar10")` from Hugging Face rather than `tf.keras.datasets.cifar10.load_data()`. The Keras helper fetches from a U Toronto mirror that may be unreliable; the HF mirror is hosted on the HF Hub CDN and stays available.

The output arrays match the standard Keras shapes — `(50000, 32, 32, 3)` uint8 for images and `(50000, 1)` for labels — so downstream code is unchanged.

In [ ]:
# Install datasets library if it's not already installed
try:
    import datasets
except ImportError:
    !pip install -q datasets
    import datasets

print("Downloading CIFAR-10 via Hugging Face datasets...")
hf_dataset = datasets.load_dataset("cifar10")
hf_dataset.set_format("numpy")

train_images = np.array(hf_dataset['train']['img'])
train_labels = np.array(hf_dataset['train']['label']).reshape(-1, 1)
test_images  = np.array(hf_dataset['test']['img'])
test_labels  = np.array(hf_dataset['test']['label']).reshape(-1, 1)

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

print("Data loaded successfully!")
print(f"Number of training samples: {len(train_images)}")
print(f"Number of test samples:     {len(test_images)}")
print(f"Image shape (raw):          {train_images.shape[1:]}")
print(f"Number of classes:          {len(class_names)}")

### Section 2: Stratified Train / Validation Split

Same procedure as the baseline (10 % held out for validation, stratified by class, then one-hot encoded). We deliberately reuse `SEED=42` so this notebook's validation set is bit-identical to the baseline's — that gives us a fair head-to-head comparison.

In [ ]:
# Keep raw uint8 — preprocessing happens inside the tf.data pipeline (Section 3)
y_train_full = train_labels.flatten()
y_test_int   = test_labels.flatten()

X_train, X_val, y_train_int, y_val_int = train_test_split(
    train_images,
    y_train_full,
    test_size=0.1,
    random_state=SEED,
    stratify=y_train_full,
)

X_test = test_images

y_train = to_categorical(y_train_int, num_classes=10)
y_val   = to_categorical(y_val_int,   num_classes=10)
y_test  = to_categorical(y_test_int,  num_classes=10)

print("Stratified split complete.")
print(f"X_train: {X_train.shape},  y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape},  y_val:   {y_val.shape}")
print(f"X_test:  {X_test.shape},  y_test:  {y_test.shape}")

### Section 3: Preprocessing Pipeline for MobileNetV2

Each image goes through three transforms inside the `tf.data` pipeline (per-batch, lazy):

1. **Cast to float32** — `preprocess_input` expects float, and `tf.image.resize` outputs float anyway.
2. **Resize 32×32 → 96×96** with bilinear interpolation. This is the minimum spatial size MobileNetV2 supports; it does not add information, but it ensures the network's stride-2 layers don't collapse the feature map prematurely.
3. **`preprocess_input`** — maps pixel range from [0, 255] to [-1, 1], matching how MobileNetV2 was pretrained on ImageNet. Order matters: do NOT pre-divide by 255.0 — `preprocess_input` expects raw [0, 255].

Augmentation (training only): mild horizontal flip + small rotation + small zoom. We keep augmentation conservative because aggressive augmentation can disturb the well-tuned pretrained features early in Stage 1.

In [ ]:
augmenter = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
], name='augmenter')

def preprocess_for_mobilenet(image, label):
    image = tf.cast(image, tf.float32)
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = preprocess_input(image)        # [-1, 1]
    return image, label

train_ds = (tf.data.Dataset.from_tensor_slices((X_train, y_train))
            .shuffle(buffer_size=10000, seed=SEED)
            .batch(BATCH_SIZE)
            .map(preprocess_for_mobilenet, num_parallel_calls=tf.data.AUTOTUNE)
            .map(lambda x, y: (augmenter(x, training=True), y),
                 num_parallel_calls=tf.data.AUTOTUNE)
            .prefetch(tf.data.AUTOTUNE))

val_ds = (tf.data.Dataset.from_tensor_slices((X_val, y_val))
          .batch(BATCH_SIZE)
          .map(preprocess_for_mobilenet, num_parallel_calls=tf.data.AUTOTUNE)
          .prefetch(tf.data.AUTOTUNE))

test_ds = (tf.data.Dataset.from_tensor_slices((X_test, y_test))
           .batch(BATCH_SIZE)
           .map(preprocess_for_mobilenet, num_parallel_calls=tf.data.AUTOTUNE)
           .prefetch(tf.data.AUTOTUNE))

# Sanity check: peek at one batch to confirm shapes and value range
for batch_x, batch_y in train_ds.take(1):
    print(f"Batch shape: {batch_x.shape}, dtype: {batch_x.dtype}")
    print(f"Pixel range: [{batch_x.numpy().min():.3f}, {batch_x.numpy().max():.3f}] (target: [-1, 1])")
    print(f"Label shape: {batch_y.shape}")

### Section 4: Build the Transfer Learning Model

Architecture — a standard transfer-learning recipe:

```
Input (96, 96, 3)
  → MobileNetV2 backbone (ImageNet, no top, ~2.26 M params, frozen for Stage 1)
  → GlobalAveragePooling2D
  → Dropout(0.3)
  → Dense(10, softmax)
```

**Important — `training=False` when calling the backbone:**
MobileNetV2 contains BatchNormalization layers whose running statistics were learned on ImageNet. We pass `training=False` when calling the backbone, which forces those BN layers to use their stored statistics regardless of whether the layer's `trainable` flag is True or False. This is the standard recipe from the Keras transfer-learning guide and prevents BN statistics from being overwritten by CIFAR's small-batch statistics during training.

In [ ]:
base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet',
)
base_model.trainable = False    # Stage 1: freeze entire backbone

inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)        # training=False keeps BN frozen
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(10, activation='softmax')(x)

model = models.Model(inputs, outputs, name='mobilenetv2_transfer_cifar10')
model.summary()

### Section 5: Stage 1 — Train the Head with Frozen Backbone

With the backbone frozen, only the new head's parameters update (`Dropout` is non-parametric, so just the final `Dense` layer's ~12 K params are trainable). This is fast and lets the head learn to map MobileNetV2's pretrained features to CIFAR's 10 classes.

**Stage 1 settings:**
- Optimizer: Adam, learning rate `1e-3` (relatively large because we're training a fresh head from scratch)
- Loss: `categorical_crossentropy`
- Epochs: 10 (head usually converges quickly)
- Callbacks: `EarlyStopping(patience=4)` to avoid overshooting

We expect Stage 1 to reach ~88–90 % validation accuracy by the end. If it's lower, the most likely cause is a preprocessing mismatch (forgetting `preprocess_input`, or running it before the resize).

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

stage1_callbacks = [
    callbacks.EarlyStopping(monitor='val_loss', patience=4,
                            restore_best_weights=True, verbose=1),
]

print("Stage 1: training the classifier head with frozen backbone")
print("-" * 65)
stage1_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=stage1_callbacks,
    verbose=1,
)

print(f"\nStage 1 finished after {len(stage1_history.history['loss'])} epoch(s).")
print(f"Best Stage 1 val accuracy: {max(stage1_history.history['val_accuracy']):.4f}")

### Section 6: Stage 2 — Fine-Tune the Top of the Backbone

We now unfreeze the **last 30 layers** of MobileNetV2. These are the deepest, most task-specific layers — they encode high-level concepts (object parts, complex textures) that benefit from being nudged toward CIFAR's specifics. The earlier layers (edges, simple textures) stay frozen to preserve the broad ImageNet prior.

**Stage 2 settings:**
- Learning rate dropped to `1e-5` (100× smaller). Fine-tuning with a large LR would shred the pretrained weights instead of refining them.
- All `BatchNormalization` layers stay in inference mode (already enforced by `training=False` at the call site, plus we explicitly mark BN layers as non-trainable for safety).
- Epochs: 20 (slow refinement; expect monotonic but small improvements per epoch).
- Callbacks add `ReduceLROnPlateau` (halve LR on plateau) and `EarlyStopping(patience=6)` for safety.

Expected outcome: an additional ~2–4 percentage points on top of Stage 1, landing at ~92–94 % validation accuracy.

In [ ]:
# Unfreeze the last 30 layers of the backbone
base_model.trainable = True
fine_tune_at = len(base_model.layers) - 30
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

# Keep all BatchNormalization layers in inference mode regardless of position
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

# Re-compile with a much smaller learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

trainable_count = sum(int(tf.size(v).numpy()) for v in model.trainable_weights)
total_count     = sum(int(tf.size(v).numpy()) for v in model.weights)
print(f"Trainable parameters: {trainable_count:,} / {total_count:,} "
      f"({100 * trainable_count / total_count:.1f} %)")
print(f"Layers unfrozen:      last 30 of {len(base_model.layers)} (BN layers stay frozen)")

stage2_callbacks = [
    callbacks.EarlyStopping(monitor='val_loss', patience=6,
                            restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', patience=3,
                                factor=0.5, min_lr=1e-7, verbose=1),
]

print("\nStage 2: fine-tuning the top of the backbone")
print("-" * 65)
stage2_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=stage1_history.epoch[-1] + 1 + 20,    # continue epoch numbering
    initial_epoch=stage1_history.epoch[-1] + 1,
    callbacks=stage2_callbacks,
    verbose=1,
)

print(f"\nStage 2 finished. "
      f"Best Stage 2 val accuracy: {max(stage2_history.history['val_accuracy']):.4f}")

### Section 7: Combined Training History (Stage 1 + Stage 2)

We merge the two `History` objects so the curves show one continuous run. The dashed vertical line marks the Stage 1 → Stage 2 boundary; you should see a small kink there as the unfrozen backbone starts contributing.

In [ ]:
# Merge stage 1 and stage 2 histories so plotting code mirrors the baseline notebook
all_keys = set(stage1_history.history) | set(stage2_history.history)
combined = {
    k: list(stage1_history.history.get(k, [])) + list(stage2_history.history.get(k, []))
    for k in all_keys
}

class _CombinedHistory:
    def __init__(self, d): self.history = d
history = _CombinedHistory(combined)

stage1_len = len(stage1_history.history['loss'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
axes[0].axvline(stage1_len - 0.5, color='red', linestyle='--', alpha=0.6,
                label='Stage 1 → 2 boundary')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].set_title('Model Accuracy (Transfer Learning)')
axes[0].legend(loc='lower right'); axes[0].grid(alpha=0.3)

axes[1].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[1].axvline(stage1_len - 0.5, color='red', linestyle='--', alpha=0.6,
                label='Stage 1 → 2 boundary')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].set_title('Model Loss (Transfer Learning)')
axes[1].legend(loc='upper right'); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Section 8: Test Set Evaluation

Same protocol as the baseline notebook so the two are directly comparable: report aggregate test loss / accuracy, then per-class precision, recall, and F1 via `classification_report`.

In [ ]:
print("Evaluating on the test set...")
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"\nTest Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}\n")

y_pred_proba = model.predict(test_ds, verbose=0)
y_pred = np.argmax(y_pred_proba, axis=1)
y_true = y_test_int

print("Per-class classification report:")
print("-" * 65)
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

### Section 9: Confusion Matrix

Compare which class pairs the transfer model still confuses against the baseline's confusion patterns. We expect the difficult animal pairs (cat/dog, deer/horse, bird/airplane) to remain the dominant confusion sources, but with reduced absolute counts.

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Count'})
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix - Transfer Learning (MobileNetV2) - CIFAR-10 Test Set')
plt.tight_layout()
plt.show()

### Section 10: High-Confidence Misclassifications

We surface the 9 test images where the model was most confident in a wrong prediction. After fine-tuning we expect these confidences to be lower than the baseline's (a healthy sign — the model has become better calibrated). Each title shows `True → Predicted (confidence)`.

In [ ]:
misclassified_mask = (y_pred != y_true)
misclassified_indices = np.where(misclassified_mask)[0]

predicted_confidence = y_pred_proba[misclassified_indices, y_pred[misclassified_indices]]
top9_local_idx = np.argsort(predicted_confidence)[-9:][::-1]
top9_global_idx = misclassified_indices[top9_local_idx]

plt.figure(figsize=(12, 12))
for i, idx in enumerate(top9_global_idx):
    plt.subplot(3, 3, i + 1)
    plt.imshow(X_test[idx])    # raw uint8, 32x32
    true_class = class_names[y_true[idx]]
    pred_class = class_names[y_pred[idx]]
    confidence = y_pred_proba[idx, y_pred[idx]]
    plt.title(f"{true_class} -> {pred_class}\n(confidence: {confidence:.1%})", fontsize=11)
    plt.axis('off')

plt.suptitle('Top 9 Most Confident Misclassifications (Transfer Learning)',
             fontsize=14, y=1.00)
plt.tight_layout()
plt.show()

### Section 11: Comparison vs. Baseline & Implementation Notes

> **Note:** Fill in actual numbers after the run.

**Headline comparison:**

| Model | Test accuracy | Trainable params | Notes |
|---|---|---|---|
| Baseline VGG-style CNN (`01_baseline_EN`) | 0.8735 | ~570 K | trained from scratch on 32×32 |
| Transfer learning (this notebook) | `[XX.X%]` | `[XX]` (Stage 2) | MobileNetV2 ImageNet → fine-tune top 30 layers |

**What worked:**
- Two-stage training kept the head training stable (Stage 1) before letting any backbone weights move (Stage 2). Skipping Stage 1 and going straight to fine-tune typically degrades the pretrained features because random head gradients propagate backwards.
- `training=False` on the backbone call site, plus explicitly freezing BN layers in Stage 2, was essential. Without it, MobileNetV2's BN statistics get overwritten by CIFAR's small-batch stats and accuracy collapses by ~10 percentage points.
- Resizing to 96×96 (the Keras-enforced minimum for MobileNetV2) was sufficient — going to 128×128 or 160×160 gives ~1 pp more at the cost of ~2× training time. Bad trade for a homework-scale experiment.

**Limitations to mention in the report:**
- **Domain gap:** CIFAR-10 images are 32×32 upsampled. The model's "input distribution" is artificial-looking compared to real photos. Performance on web-scraped 200×200+ images will degrade visibly. This is what Phase 3A (`04_realworld_testing`) tests.
- The fine-tune set (last 30 layers) was chosen by convention, not by hyperparameter search. A study of "how many layers to unfreeze" could be a small ablation in a follow-up.

### Section 12: Save Trained Model and Training History

Persist artifacts to Google Drive for downstream notebooks (`04_realworld_testing`) and for the Streamlit app to load.

- `transfer_mobilenet_v1.keras` — full Keras v3 archive (architecture + weights + optimizer state)
- `transfer_mobilenet_v1_history.pkl` — combined Stage 1 + Stage 2 history dict (portable across TF versions)

In [ ]:
import pickle, os

model_path = f"{MODEL_DIR}/transfer_mobilenet_v1.keras"
model.save(model_path)
print(f"Model saved:   {model_path}  ({os.path.getsize(model_path) / 1e6:.1f} MB)")

history_path = f"{MODEL_DIR}/transfer_mobilenet_v1_history.pkl"
with open(history_path, "wb") as f:
    pickle.dump(history.history, f)
print(f"History saved: {history_path}")